In [2]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time

In [3]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

In [4]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

In [32]:
df_income_brackets = pd.read_excel(os.path.join(path_config0, 'CA State Income Brackets by Household Size.xlsx'), sheet_name = 'Table')
df_income_brackets['County'].fillna(method='ffill', inplace = True)
df_income_brackets['County'] = df_income_brackets['County'].str.replace(' County.*', '', regex = True)
df_income_brackets['County'] = df_income_brackets['County'].str.replace('\n', ' ', regex = True)
df_income_brackets['AMI'   ] = df_income_brackets['County'].str.extract('\$?([0-9,]+)[.%]?')
df_income_brackets['AMI'   ] = df_income_brackets['AMI'   ].str.replace(',', '', regex = True)
df_income_brackets['County'] = df_income_brackets['County'].str.replace(' \$?([0-9,]+)[.%]?', '', regex = True)
df_income_brackets = pd.melt(df_income_brackets
                              , id_vars = ['County', 'Income Bracket', 'AMI']
                              , var_name = 'NP'
                              , value_name = 'Income Threshold'
                            )
df_income_brackets = df_income_brackets[df_income_brackets['Income Bracket'].isin(['Low Income', 'Moderate Income'])]
df_income_brackets

,County,Income Bracket,AMI,NP,Income Threshold
3,Alameda,Low Income,147900,1,78550
5,Alameda,Moderate Income,147900,1,124250
9,Alpine,Low Income,114600,1,53850
11,Alpine,Moderate Income,114600,1,96250
15,Amador,Low Income,101200,1,51350
...,...,...,...,...,...
2771,Ventura,Moderate Income,123500,8,195600
2775,Yolo,Low Income,114000,8,110750
2777,Yolo,Moderate Income,114000,8,180600
2781,Yuba,Low Income,83800,8,87100
